In [ ]:
import os
print(os.getcwd())

In [1]:
import pandas as pd
import re

jd = pd.read_csv("../data/raw/jd_full_text_20260811.csv")
meta = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")

df = jd.merge(meta, on="id", how="left")
df["jd_lower"] = df["jd_text"].str.lower()
df["title_lower"] = df["title"].str.lower()
print(df.shape)

(214, 17)


In [2]:
SKILLS = {
    "cloud": ["aws", "s3", "ec2", "lambda", "sagemaker", "redshift",
              "athena", "azure", "gcp", "bigquery", "databricks",
              "snowflake", "terraform", "kubernetes", "docker"],
    "language": ["python", "r", "sql", "scala", "java", "c++",
                 "matlab", "sas", "julia", "fortran"],
    "library": ["pandas", "numpy", "scikit-learn", "pyspark", "spark",
                "airflow", "tensorflow", "pytorch", "keras", "git"],
    "bi": ["excel", "power bi", "tableau", "looker", "vba"],
    "method": ["machine learning", "deep learning", "nlp", "statistics",
               "a/b testing", "forecasting", "etl", "mlops",
               "numerical methods", "hpc", "parallel computing",
               "optimisation", "simulation", "rag", "vector database",
               "embeddings"],
}

all_skills = [s for group in SKILLS.values() for s in group]
print(len(all_skills))

df["jd_lower"] = df["jd_text"].str.lower()
print(df["jd_lower"].iloc[0][:200])

56
about us: lseg (london stock exchange group) is more than a diversified global financial markets infrastructure and data business. we are dedicated, open-access partners with a dedication to excellenc


In [3]:
naive = {}
for s in all_skills:
    naive[s] = df["jd_lower"].str.contains(s, regex=False).sum()

res = pd.Series(naive).sort_values(ascending=False)
print(res.head(20))

r                   214
excel               127
python              108
rag                 102
machine learning     87
sql                  78
git                  76
scala                72
statistics           52
aws                  43
azure                35
power bi             29
gcp                  26
pytorch              24
optimisation         23
tableau              23
mlops                23
forecasting          21
docker               21
tensorflow           18
dtype: int64


In [4]:
import re

def build_pattern(skill):
    """把技能词转成带词边界的正则"""
    return r"\b" + re.escape(skill) + r"\b"

for s in ["r", "rag", "scala", "excel", "c++", "a/b testing", "power bi"]:
    print(f"{s:15s} {build_pattern(s)}")

r               \br\b
rag             \brag\b
scala           \bscala\b
excel           \bexcel\b
c++             \bc\+\+\b
a/b testing     \ba/b\ testing\b
power bi        \bpower\ bi\b


In [ ]:
counts = {}
for s in all_skills:
    p = build_pattern(s)
    counts[s] = df["jd_lower"].str.contains(p, regex=True).sum()

res2 = pd.Series(counts).sort_values(ascending=False)
print(res2.head(25))

In [ ]:
def show_context(skill, n=8, width=60):
    """打印某技能词命中处的前后文"""
    p = build_pattern(skill)
    hits = df[df["jd_lower"].str.contains(p, regex=True)]
    print(f"=== {skill}  命中 {len(hits)} 条，抽 {min(n, len(hits))} 条 ===")
    for t in hits["jd_lower"].head(n):
        m = re.search(p, t)
        s = max(0, m.start() - width)
        e = min(len(t), m.end() + width)
        print("…" + t[s:e].replace("\n", " ") + "…")
    print()

for s in ["r", "excel", "git", "java"]:
    show_context(s)

In [5]:
def build_pattern(skill):
    """带词边界的正则；对已知歧义词做特殊处理"""
    special = {
        # R：排除 r&d
        "r": r"\br\b(?!\s*&\s*d\b)",
        # excel：排除动词用法 "excel at/in/if/when"
        "excel": r"\bexcel\b(?!\s+(?:at|in|if|when)\b)",
        # git：纳入 github、gitlab
        "git": r"\bgit(?:hub|lab)?\b",
        # sql：纳入 postgresql、mysql、nosql 等变体
        "sql": r"\b(?:my|postgre|no|t-|pl/)?sql\b",
    }
    if skill in special:
        return special[skill]
    return r"\b" + re.escape(skill) + r"\b"

for s in ["r", "excel", "git", "sql", "python"]:
    print(f"{s:10s} {build_pattern(s)}")

r          \br\b(?!\s*&\s*d\b)
excel      \bexcel\b(?!\s+(?:at|in|if|when)\b)
git        \bgit(?:hub|lab)?\b
sql        \b(?:my|postgre|no|t-|pl/)?sql\b
python     \bpython\b


In [6]:
df["title_lower"] = df["title"].str.lower()

SENIORITY = [
    ("graduate", r"\b(graduate|intern|internship|placement|trainee|apprentice)\b"),
    ("junior",   r"\b(junior|entry.level|jr\.?)\b"),
    ("senior",   r"\b(senior|snr\.?|sr\.?)\b"),
    ("lead",     r"\b(lead|principal|staff|head\s+of|director|chief)\b"),
]

def get_seniority(t):
    for label, pat in SENIORITY:
        if re.search(pat, t):
            return label
    return "unspecified"

df["seniority"] = df["title_lower"].apply(get_seniority)
print(df["seniority"].value_counts())

seniority
unspecified    104
senior          41
graduate        41
lead            25
junior           3
Name: count, dtype: int64


In [7]:
for lab in ["graduate", "junior", "lead"]:
    print(f"=== {lab} ===")
    print(df[df["seniority"] == lab]["title"].head(10).to_string(index=False))
    print()

=== graduate ===
                           Graduate Water Modeller
                         Graduate Business Analyst
Graduate Geospatial (GIS) Consultant - Communit...
                           Graduate Data Scientist
                      Analytics Graduate Programme
                         Data and Analytics Intern
           Data Analyst Intern - Financial Markets
                             Graduate Data Analyst
Data Analyst Placement Programme No Experience ...
                             Graduate Data Analyst

=== junior ===
Junior Data Analyst – Tech Startup
Junior English Linguistics Analyst
Junior English Linguistics Analyst

=== lead ===
                          Principal Data Scientist
        Staff Software Engineer - Machine Learning
                          Principal Data Scientist
                       Lead Product Manager (Data)
                    Lead Machine Learning Engineer
                               Lead Data Scientist
                   Staff Machine

In [ ]:
print(df[df["title_lower"].str.contains("junior|jr|entry", regex=True)]["title"].to_string(index=False))
print("---")
print(df[df["title_lower"].str.contains("manager", regex=True)]["title"].to_string(index=False))

In [ ]:
meta_all = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")
t = meta_all["title"].str.lower()

for pat in ["junior", "jr", "entry", "graduate", "trainee", "placement"]:
    n = t.str.contains(r"\b" + pat + r"\b", regex=True).sum()
    print(f"{pat:12s} {n:>4d}  ({n/len(meta_all)*100:.1f}%)")

In [8]:
for s in all_skills:
    df["has_" + s] = df["jd_lower"].str.contains(build_pattern(s), regex=True)

skill_cols = ["has_" + s for s in all_skills]

grouped = df.groupby("seniority")[skill_cols].mean().T
grouped.columns = [f"{c}(n={(df['seniority']==c).sum()})" for c in grouped.columns]
grouped["overall"] = df[skill_cols].mean()

top = grouped.sort_values("overall", ascending=False).head(20)
print((top * 100).round(1).to_string())

                      graduate(n=41)  junior(n=3)  lead(n=25)  senior(n=41)  unspecified(n=104)  overall
has_python                      34.1          0.0        52.0          63.4                51.0     49.5
has_machine learning            29.3          0.0        52.0          43.9                41.3     40.2
has_sql                         34.1          0.0        20.0          48.8                36.5     36.0
has_statistics                  22.0         33.3        16.0          29.3                25.0     24.3
has_azure                        4.9          0.0        12.0          26.8                18.3     16.4
has_aws                          9.8          0.0        16.0          22.0                17.3     16.4
has_power bi                    29.3          0.0         8.0           4.9                12.5     13.6
has_gcp                          4.9          0.0        16.0          19.5                11.5     12.1
has_r                           12.2          0.0      

In [9]:
df.to_csv("../data/processed/jobs_with_skills_20260810.csv", index=False)
grouped.to_csv("../data/processed/skill_by_seniority_20260810.csv")
print(df.shape)

(214, 74)
